In [ ]:
# papermill parameters cell — injected at runtime
sample_id = ""
session_dir = ""
out_dir = ""
dataset_label = ""
vbd_current_threshold_a = 1e-2
vbd_compliance_fraction = 0.5
title = ""


In [ ]:
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out_path = Path(out_dir)
out_path.mkdir(parents=True, exist_ok=True)
session_path = Path(session_dir)

plot_title = title if title else sample_id
print(f"Session : {session_dir}")
print(f"Output  : {out_dir}")
print(f"Label   : {dataset_label}")
print(f"Vbd thr : {vbd_current_threshold_a:.2e} A")

In [ ]:
# Discover all VAC devices in session
# Layout: <session>/<device>/data/VAC_*.data

SIZE_ORDER = ['big', 'mid', 'small', 'little', 'tiny']

def size_from_name(device_name):
    raw = device_name.split('_')[0]
    return 'little' if raw == 'littlel' else raw


def most_complete_vac_file(vac_files):
    """Pick the VAC file with the most data rows (most complete measurement)."""
    if len(vac_files) == 1:
        return vac_files[0]
    best, best_rows = vac_files[0], 0
    for f in vac_files:
        try:
            n = sum(1 for _ in f.open()) - 1  # subtract header
        except Exception:
            n = 0
        if n > best_rows:
            best, best_rows = f, n
    return best


vac_devices = []  # list of dicts: {name, size, vac_file}

for dev_dir in sorted(session_path.iterdir()):
    if not dev_dir.is_dir():
        continue
    data_dir = dev_dir / 'data'
    if not data_dir.is_dir():
        continue
    vac_files = sorted(data_dir.glob('VAC_*.data'))
    if not vac_files:
        print(f"  SKIP {dev_dir.name}: no VAC_*.data")
        continue
    # Pick the most complete file (most data rows)
    chosen = most_complete_vac_file(vac_files)
    if len(vac_files) > 1:
        print(f"  {dev_dir.name}: {len(vac_files)} VAC files — chose {chosen.name} (most complete)")
    vac_devices.append({
        'name': dev_dir.name,
        'size': size_from_name(dev_dir.name),
        'vac_file': chosen,
    })

print(f"Found {len(vac_devices)} VAC devices:")
for d in vac_devices:
    print(f"  {d['name']} — {d['vac_file'].name}")


In [ ]:
# Load each device, extract positive sweep, detect Vbd
# Vbd = first V where |I| >= vbd_compliance_fraction * compliance_current_a
# (read from .meta.json next to the .data file; fallback threshold = 0.1 A)


def read_compliance_current(vac_file):
    """Read compliance_current_a from .meta.json next to vac_file. Returns None if absent."""
    meta_file = vac_file.with_suffix('.meta.json')
    if not meta_file.exists():
        return None
    try:
        meta = json.loads(meta_file.read_text())
        return float(meta['parameters']['compliance_current_a'])
    except Exception:
        return None


for dev in vac_devices:
    df = pd.read_csv(dev['vac_file'])
    # Positive sweep: Direction == 1 AND Voltage >= 0
    pos = df[(df['Direction'] == 1) & (df['Voltage'] >= 0)].copy()
    pos = pos.sort_values('Voltage').reset_index(drop=True)

    V = pos['Voltage'].to_numpy(dtype=float)
    I = pos['Current'].to_numpy(dtype=float)
    I_abs = np.abs(I)

    dev['V'] = V
    dev['I_abs'] = I_abs

    # Determine effective threshold
    compliance_a = read_compliance_current(dev['vac_file'])
    fallback_thr = 0.1  # A — used when meta is absent or compliance threshold not reached
    if compliance_a is not None:
        compliance_thr = vbd_compliance_fraction * compliance_a
        # If compliance threshold exceeds data range, fall back to fixed threshold
        effective_thr = compliance_thr if I_abs.max() >= compliance_thr else fallback_thr
        dev['compliance_a'] = compliance_a
        dev['compliance_thr'] = compliance_thr
    else:
        effective_thr = fallback_thr
        dev['compliance_a'] = None
        dev['compliance_thr'] = None
    dev['effective_thr'] = effective_thr

    # Vbd = first voltage where |I| >= effective threshold
    mask = I_abs >= effective_thr
    if mask.any():
        idx = int(np.argmax(mask))
        dev['Vbd'] = float(V[idx])
        dev['Vbd_I'] = float(I_abs[idx])
    else:
        # Last-resort fallback: voltage at maximum |I|
        dev['Vbd'] = float(V[int(np.argmax(I_abs))])
        dev['Vbd_I'] = float(I_abs.max())
        print(f"  WARNING {dev['name']}: threshold {effective_thr:.2e} A not reached; using V at max|I|={dev['Vbd']:.2f} V")

    print(f"{dev['name']}: Vbd={dev['Vbd']:.2f} V  thr={effective_thr:.2e} A  (max|I|={I_abs.max():.3e} A)")


In [ ]:
vbds = [d['Vbd'] for d in vac_devices]
vbd_min = float(np.min(vbds))
vbd_max = float(np.max(vbds))
vbd_avg = float(np.mean(vbds))

label = dataset_label if dataset_label else sample_id
dataset_color = 'steelblue'

fig, ax = plt.subplots(figsize=(9, 6))

for dev in vac_devices:
    V = dev['V']
    I_abs = dev['I_abs']
    # Replace zeros with NaN for log plot
    I_plot = np.where(I_abs > 0, I_abs, np.nan)
    ax.semilogy(V, I_plot, color=dataset_color, alpha=0.3, linewidth=0.9)
    # Mark Vbd at the actual current value at that point
    ax.plot(dev['Vbd'], dev['Vbd_I'], marker='^',
            color=dataset_color, markersize=7, zorder=5)

# Dashed vertical line at avg Vbd
ax.axvline(vbd_avg, color=dataset_color, linestyle='--', linewidth=1.5)
# Shaded band [min, max]
ax.axvspan(vbd_min, vbd_max, color=dataset_color, alpha=0.08)

# Annotation text
ann_text = f"{label}: Vbd [{vbd_min:.2f}, {vbd_max:.2f}] V, avg {vbd_avg:.2f} V"
ax.text(0.05, 0.08, ann_text, transform=ax.transAxes,
        color=dataset_color, fontsize=10, fontweight='bold',
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

# Legend entry for dataset
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color=dataset_color, marker='^', markersize=8,
           label=f"{label}  (n={len(vac_devices)})"),
]
ax.legend(handles=legend_elements, title='Dataset (color)', loc='upper right', fontsize=9)

ax.set_xlabel('Bias V (V)', fontsize=12)
ax.set_ylabel('Current I (A)', fontsize=12)
ax.set_xlim(left=0, right=5)
ax.set_ylim(bottom=1e-10)
ax.grid(True, which='both', alpha=0.3)

if plot_title:
    ax.set_title(plot_title, fontsize=11)

fig.tight_layout()

out_png = out_path / 'iv_breakdown_summary.png'
fig.savefig(str(out_png), dpi=130, format='png', bbox_inches='tight')
plt.close(fig)
print(f"Saved: {out_png}")
print(f"Vbd min={vbd_min:.2f} max={vbd_max:.2f} avg={vbd_avg:.2f} V  n={len(vac_devices)}")


In [ ]:
metrics = {
    'vbd_min': vbd_min,
    'vbd_max': vbd_max,
    'vbd_avg': vbd_avg,
    'n_devices': len(vac_devices),
    'dataset_label': label,
    'vbd_compliance_fraction': vbd_compliance_fraction,
    'vbd_current_threshold_a': vbd_current_threshold_a,
}

metrics_path = out_path / 'metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"Metrics written to {metrics_path}")
print(json.dumps(metrics, indent=2))
